In [7]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1" 
import cv2
import time
import torch
import datetime
import numpy as np
from torch import nn

from torchvision import models
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd

from pytorch_grad_cam import GradCAM, \
    HiResCAM, \
    ScoreCAM, \
    GradCAMPlusPlus, \
    AblationCAM, \
    XGradCAM, \
    EigenCAM, \
    EigenGradCAM, \
    LayerCAM, \
    FullGrad, \
    FinerCAM, \
    KPCACAM, \
    GradCAMElementWise

from pytorch_grad_cam import GuidedBackpropReLUModel
from pytorch_grad_cam.ablation_cam_new import AblationCAMNEW
from pytorch_grad_cam.grad_cam_new import Grad_CAM_NEW
from pytorch_grad_cam.layer_cam_new import Layer_CAM_NEW
from pytorch_grad_cam.score_cam_new import ScoreCAMNew
from pytorch_grad_cam.mgrad_cam import MGradCAM

from pytorch_grad_cam.utils.image import show_cam_on_image, \
    deprocess_image, \
    preprocess_image, show_image

from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget, ClassifierOutputSigmoidTarget

In [8]:
methods = {"gradcam": GradCAM,
        "hirescam": HiResCAM,
        "scorecam": ScoreCAM,
        "scorecamnew": ScoreCAMNew,
        "gradcam++": GradCAMPlusPlus,
        "ablationcam": AblationCAM,
        "ablationcamnew": AblationCAMNEW,
        "xgradcam": XGradCAM,
        "eigencam": EigenCAM,
        "eigengradcam": EigenGradCAM,
        "layercam": LayerCAM,
        "fullgrad": FullGrad,
        "gradcamelementwise": GradCAMElementWise,
        "layercamnew": Layer_CAM_NEW,
        "gradcamnew": Grad_CAM_NEW,
        "finercam": FinerCAM,
        "kpcacam": KPCACAM,
        "mgradcam": MGradCAM}

In [9]:
exdark_class = ['bicycle', 'boat', 'bottle','bus', 
                'car', 'cat', 'chair', 'cup', 
                'dog', 'motorbike', 'people', 'table']

In [10]:
path = r"/devdata/home/homefun"

In [11]:
target = [0, 10]
# target = [8]
image_path = os.path.join(path, r"DATA/ExDark/images/Bicycle/2015_00173.jpg")
save_path = os.path.join(path, r"CAM-copy/pic_result_new/exdark_InsDel")
resume = os.path.join(path, r"weights/coco_cam/exdark_loss_20250911085934/best_acc.pth")
aug_smooth = False
eigen_smooth = False
use_cuda = True if torch.cuda.is_available() else False
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")

In [12]:
model = models.resnet50(weights=None)
model.fc = nn.Linear(2048, 12)
model.load_state_dict(torch.load(resume, map_location='cpu')['model'])

# target_layers = [model.layer4]
# target_layers = [model.layer2, model.layer3, model.layer4]

rgb_img = cv2.imread(image_path)[:, :, ::-1]
rgb_img = cv2.resize(rgb_img, [224, 224])
rgb_img = np.float32(rgb_img) / 255
input_tensor = preprocess_image(rgb_img,
                                mean=[0.485, 0.456, 0.406],
                                std=[0.229, 0.224, 0.225])
targets = [ClassifierOutputTarget(int(tar)) for tar in target]
input_tensor = torch.cat([input_tensor for tar in target], dim=0)

model, input_tensor = model.eval().to(device), input_tensor.to(device)
y = torch.sigmoid(model(input_tensor))[0, target]
print(torch.sigmoid(model(input_tensor))[0, target])

In [13]:
def process(method = "layercam", target_layers = [model.layer4]):
    cam_algorithm = methods[method]
    cam = cam_algorithm(model=model, target_layers=target_layers, use_cuda=use_cuda)
    # AblationCAM and ScoreCAM have batched implementations.
    # You can override the internal batch size for faster computation.
    cam.batch_size = 32
    # Finer-CAM的时候需要用到device，其它CAM类方法可以注释掉
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    if hasattr(cam, 'device'):
        cam.device = device
    else:
        # 如果CAM没有device属性，手动设置
        cam.device = device
    # 对于某些CAM方法，可能需要手动设置activations_and_gradients的设备
    if hasattr(cam, 'activations_and_grads'):
        cam.activations_and_grads.device = device
    grayscale_cams = cam(input_tensor=input_tensor,
                            targets=targets,
                            aug_smooth=aug_smooth,
                            eigen_smooth=eigen_smooth)
    return grayscale_cams

In [14]:
def getauc(p, index, target):
    # print(os.path.join(path, 'weights', p, 'record.csv'))
    csv = pd.read_csv(os.path.join(path, 'weights', p, 'record.csv'))
    result = []
    # print(csv.columns)
    if index == 'delete':
        percentiles = [i for i in range(100, 0, -1)]
    else:
        percentiles = [i for i in range(99, -1, -1)]
    for i in percentiles:
        result.append(csv.loc[(csv['image_id'] == image_path.split('/')[-1])
                              & (csv['label'] == target), index + '_' + str(i)].values[0])
    return result

In [15]:
def savepic(grayscale_cams, method, layers):
    if not os.path.exists(os.path.join('exdark_InsDel', method)):
        os.makedirs(os.path.join('exdark_InsDel', method))
    for index, grayscale_cam in enumerate(grayscale_cams):
        cam_image = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)
        image = Image.fromarray(cam_image.astype(np.uint8))
        image.save(os.path.join('exdark_InsDel', method,
                                image_path.split('/')[-1].split('.')[0] + '_' + layers + '_' + exdark_class[target[index]] + '.pdf'), 'PDF', resolution=100.0, save_all=True)
        image.save(os.path.join('exdark_InsDel', method,
                                image_path.split('/')[-1].split('.')[0] + '_' + layers + '_' + exdark_class[target[index]] + '.png'))
    image = Image.fromarray((rgb_img * 255).astype(np.uint8))
    image.save(os.path.join('exdark_InsDel', method,
                                 image_path.split('/')[-1].split('.')[0] + '.pdf'), 'PDF', resolution=100.0, save_all=True)
    image.save(os.path.join('exdark_InsDel', method,
                                 image_path.split('/')[-1].split('.')[0] + '.png'))

In [16]:
def compare(camdict, target_layers = [model.layer4]):
    recordDict = {}
    for method, p in camdict.items():
        grayscale_cams = process(method, target_layers)
        savepic(grayscale_cams, method, '432' if len(target_layers) > 1 else '4')
    for t in target:
        insdict = {}
        deldict = {}
        for method, p in camdict.items():
            insdict[method] = getauc(p, 'insert', t)
            deldict[method] = getauc(p, 'delete', t)
        print(len(insdict), len(deldict))
        recordDict[t] = {'ins' : insdict, 'del' : deldict}
    return recordDict

In [17]:
def main(method = "layercam", target_layers = [model.layer4], save=False, save_name='432'):
    grayscale_cams = process(method, target_layers)
    # Here grayscale_cam has only one image in the batch
    for index, grayscale_cam in enumerate(grayscale_cams):
        plt.subplot(2, 2, index + 1)
        cam_image = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)
        if save:
            image = Image.fromarray(cam_image.astype(np.uint8))
            image.save(os.path.join('exdark_InsDel', method,
                        image_path.split('/')[-1].split('.')[0] + '_' + save_name + '_' + exdark_class[target[index]] + '.png'))
        plt.imshow(cam_image)
    plt.tight_layout()
    plt.show()

In [18]:
camdict = { 
            'gradcam'   : r"Exdark_CAM/gradcam/4/202511210005",
            'gradcam++' : r"Exdark_CAM/gradcam++/4/202509120006",  
            'layercam'  : r"Exdark_CAM/layercam/4/202509120123",  
            'fullgrad'  : r"Exdark_CAM/fullgrad/4/202509121018",  
            # # 'mgradcam'  : r"CAM/mgradcam/4/202312062043",
            'scorecam'  : r"Exdark_CAM/scorecam/4/202509130629",  
            'ablationcam'  : r"Exdark_CAM/ablationcam/4/202509120323",  
            # 'xgradcam'  : r"Exdark_CAM/xgradcam/4/202509121630",  
            # 'eigencam'  : r"Exdark_CAM/eigencam/4/202509121824",  
            'kpcacam'  : r"Exdark_CAM/kpcacam/4/202509122037", 
            'finercam'  : r"Exdark_CAM/finercam/4/202512010021",  
            # 'eigengradcam'  : r"Exdark_CAM/eigengradcam/4/202509120209",
            'mgradcam'  : r"Exdark_CAM/mgradcam/432/202511210959",
            }
recordDict = compare(camdict)
recordcsv = pd.DataFrame(columns=list(range(103)))
for k, v in recordDict.items():
    for k1, v1 in v['del'].items():
        l = ['del', exdark_class[k], k1]
        l.extend(v1)
        recordcsv.loc[len(recordcsv)] = l
for k, v in recordDict.items():
    for k1, v1 in v['ins'].items():
        l = ['ins', exdark_class[k], k1]
        l.extend(v1)
        recordcsv.loc[len(recordcsv)] = l

save_dir = "exdark_InsDel"
save_file = os.path.join(save_dir, "auc.xlsx")
os.makedirs(save_dir, exist_ok=True)

print("当前工作目录:", os.getcwd())
print("recordDict keys:", recordDict.keys())
print("recordcsv shape:", recordcsv.shape)
print("将要保存的路径:", os.path.abspath(save_file))

recordcsv.to_excel(save_file, index=False)
print("✅ 保存成功:", os.path.abspath(save_file))

curve_dir = os.path.join(save_dir, "curves")
os.makedirs(curve_dir, exist_ok=True)

for t, results in recordDict.items():
    for metric in ["ins", "del"]:
        plt.figure(figsize=(6, 4))
        colors = plt.cm.get_cmap('tab20', len(results[metric]))
        for idx, (method, values) in enumerate(results[metric].items()):
            plt.plot(range(len(values)), values, label=method, color=colors(idx))
        # plt.xlabel(f"{metric.upper()} point")
        # plt.ylabel("Class Score")
        plt.title(f"{exdark_class[t]}")
        plt.legend()
        out_path = os.path.join(curve_dir, f"{metric}_{exdark_class[t]}.png")
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"✅ 曲线已保存: {os.path.abspath(out_path)}")

In [19]:
main("mgradcam")

In [20]:
main("mgradcam", [model.layer2, model.layer3, model.layer4])

In [21]:
main("gradcam")

In [22]:
main("gradcam++")

In [23]:
main("layercam")

In [24]:
main("fullgrad")

In [25]:
main("finercam")

In [26]:
main("scorecam")

In [27]:
main("ablationcam")

In [28]:
main("xgradcam")

In [29]:
main("eigencam")

In [30]:
main("kpcacam")

In [31]:
main("eigengradcam")